In [1]:
# Fixed settings

NOTEBOOK_VERSION = "part1b-v2-fresh-multilingual-enrichment"
SEED = 42

EXPECTED_FROZEN_ROWS = 6620
EXPECTED_FROZEN_SHA256 = "0cbfba6ea4c3e1bd84bc1fed31c364ab899e9c6a8e68314e10825bc070e74380"
EXPECTED_ADDITION_WORDS = ["vana", "yekuroja", "yekurara", "kutengeswa", "ndinoda"]
EXPECTED_ADDITION_COUNT = 5

# Fresh translation configuration
GOOGLE_MAX_RETRIES = 4
GOOGLE_SLEEP_SECONDS = 0.75
OPENAI_SECRET_NAME = "OPENAI_API_KEY"
OPENAI_MODEL = "gpt-5-mini"
OPENAI_MAX_RETRIES = 4
OPENAI_SLEEP_SECONDS = 1.0

# Google language identifiers. Runtime support is recorded before translation.
GOOGLE_TARGETS = {
    "English": {"preferred_code": "en", "names": ["english"]},
    "French": {"preferred_code": "fr", "names": ["french"]},
    "Zulu": {"preferred_code": "zu", "names": ["zulu"]},
    "Afrikaans": {"preferred_code": "af", "names": ["afrikaans"]},
    "Sepedi": {"preferred_code": "nso", "names": ["sepedi", "northern sotho"]},
    "Xhosa": {"preferred_code": "xh", "names": ["xhosa"]},
}
SHONA_LANGUAGE = {"preferred_code": "sn", "names": ["shona"]}
CILUBA_RUNTIME_NAMES = [
    "ciluba", "tshiluba", "luba-kasai", "luba kasai", "luba-katanga", "luba katanga"
]
CILUBA_RUNTIME_CODES_TO_TRY = ["lua", "tsh", "lu"]

# Legacy-resource matching is comparison evidence only.
LEGACY_MATCHES_PER_WORD = 8
LEGACY_STRONG_MATCH_THRESHOLD = 88.0

# Only these inherited schema fields may be populated for the five additions.
ENRICHMENT_FIELDS = [
    "CILUBA", "French", "Nature", "English", "Zulu", "Afrikaans", "Sepedi", "Xhosa"
]
CORE_SCHEMA_COLUMNS = [
    "CILUBA", "French", "Score", "Sentiment", "Nature", "English", "Zulu",
    "Afrikaans", "Sepedi", "Xhosa", "Shona", "expanded_shona_class", "expanded_shona"
]

# Leave None on Kaggle. It can be set to a local directory for controlled testing.
INPUT_ROOT_OVERRIDE = None
OUTPUT_ROOT = "/kaggle/working/FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment"
RESET_OUTPUT_ROOT = True

In [2]:
# Imports, packages, paths and reusable utilities

import sys
import os
import re
import json
import time
import math
import shutil
import hashlib
import zipfile
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd


def ensure_package(import_name, pip_spec):
    try:
        return __import__(import_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_spec])
        return __import__(import_name)


openpyxl = ensure_package("openpyxl", "openpyxl>=3.1")
deep_translator = ensure_package("deep_translator", "deep-translator==1.11.4")
rapidfuzz = ensure_package("rapidfuzz", "rapidfuzz>=3.9")
openai_module = ensure_package("openai", "openai>=1.40")

import matplotlib.pyplot as plt
from deep_translator import GoogleTranslator
from rapidfuzz import fuzz
from openai import OpenAI

np.random.seed(SEED)


def select_output_root(configured):
    configured_path = Path(configured).expanduser()
    candidates = [
        configured_path,
        Path.cwd() / "FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment",
        Path("/mnt/data/FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment"),
        Path.home() / "FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment",
    ]
    errors = []
    for candidate in candidates:
        try:
            candidate.parent.mkdir(parents=True, exist_ok=True)
            probe = candidate.parent / f".write_probe_{os.getpid()}"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
    raise RuntimeError("No writable output directory. Tried:\n- " + "\n- ".join(errors))


ROOT_OUT = select_output_root(OUTPUT_ROOT)
unsafe_roots = {
    Path("/").resolve(strict=False),
    Path.home().resolve(strict=False),
    Path.cwd().resolve(strict=False),
    Path("/kaggle/working").resolve(strict=False),
    Path("/mnt/data").resolve(strict=False),
}
if RESET_OUTPUT_ROOT and ROOT_OUT.exists():
    if ROOT_OUT.resolve(strict=False) in unsafe_roots:
        raise RuntimeError(f"Refusing to reset unsafe output path: {ROOT_OUT}")
    shutil.rmtree(ROOT_OUT)
ROOT_OUT.mkdir(parents=True, exist_ok=True)

TAB = ROOT_OUT / "tables"
FIG = ROOT_OUT / "figures"
AUD = ROOT_OUT / "audit"
for directory in (TAB, FIG, AUD):
    directory.mkdir(parents=True, exist_ok=True)

# Persistent caches remain outside the results directory.
CACHE_ROOT = ROOT_OUT.parent / "FrenchyShona_V2_Part1B_Fresh_Cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TRANSLATION_CACHE_PATH = CACHE_ROOT / "google_translation_cache.csv"
CILUBA_CACHE_PATH = CACHE_ROOT / "ciluba_gpt_cache.csv"
INPUT_CACHE_ROOT = ROOT_OUT.parent / "FrenchyShona_V2_Part1B_InputCache"
INPUT_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalise_text(value):
    if pd.isna(value):
        return ""
    text = str(value).replace("\u00a0", " ").strip().casefold()
    return re.sub(r"\s+", " ", text)


def normalise_compare(value):
    text = normalise_text(value)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def nonblank(value):
    return bool(normalise_text(value))


def read_any(path):
    suffix = path.suffix.casefold()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    raise ValueError(f"Unsupported table type: {path}")


def safe_string(value):
    return "" if pd.isna(value) else str(value).strip()


def parse_json_object(text):
    if not isinstance(text, str):
        raise ValueError("Model response is not text.")
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.I)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        value = json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.S)
        if not match:
            raise
        value = json.loads(match.group(0))
    if not isinstance(value, dict):
        raise ValueError("Expected one JSON object.")
    return value


input_root = Path(INPUT_ROOT_OVERRIDE).expanduser() if INPUT_ROOT_OVERRIDE else Path("/kaggle/input")
if not input_root.exists():
    fallback = Path("/mnt/data")
    if fallback.exists():
        input_root = fallback
    else:
        raise FileNotFoundError("Neither the configured input root nor /mnt/data exists.")

# Extract attached ZIPs. The notebook works whether Kaggle exposes archives or extracted files.
extract_root = INPUT_CACHE_ROOT / "extracted_inputs"
extract_root.mkdir(parents=True, exist_ok=True)
search_roots = [input_root]
for archive_path in sorted(input_root.rglob("*.zip")):
    if not archive_path.is_file():
        continue
    try:
        archive_hash = sha256_file(archive_path)[:16]
        destination = extract_root / f"{archive_path.stem}_{archive_hash}"
        if not destination.exists():
            destination.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archive_path) as archive:
                archive.extractall(destination)
        search_roots.append(destination)
    except (zipfile.BadZipFile, OSError):
        continue


def find_files_by_name(filename):
    candidates = []
    for root in search_roots:
        candidates.extend(root.rglob(filename))
    unique_by_hash = {}
    for path in candidates:
        if path.is_file():
            unique_by_hash.setdefault(sha256_file(path), path)
    return list(unique_by_hash.values())


def require_unique_file(filename):
    matches = find_files_by_name(filename)
    if not matches:
        raise FileNotFoundError(
            f"Required file {filename!r} was not found in the attached Kaggle inputs."
        )
    if len(matches) > 1:
        details = "\n".join(f"- {p} ({sha256_file(p)})" for p in matches)
        raise RuntimeError(
            f"Multiple non-identical copies of {filename!r} were found. "
            f"Remove the divergent input:\n{details}"
        )
    return matches[0]


print("Input root:", input_root)
print("Output folder:", ROOT_OUT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.1 MB/s eta 0:00:00
Input root: /kaggle/input
Output folder: /kaggle/working/FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment


In [3]:
# Locate the two required input packages and verify the frozen V7.4 resource

FROZEN_PATH = require_unique_file("FrenchyShona_V2_v7_4_frozen.csv")
ADDITIONS_PATH = require_unique_file("validated_additions_v7_4.csv")
CONTEXT_PATH = require_unique_file("rag_contextual_candidate_assessment_complete.csv")
EXPANDED_6632_PATH = require_unique_file("expanded_lexicon.xlsx")
ORIGINAL_6963_PATH = require_unique_file("lexicon_6000 words.xlsx")

input_files = {
    "v7_4_frozen": FROZEN_PATH,
    "v7_4_additions": ADDITIONS_PATH,
    "v7_4_contextual_assessment": CONTEXT_PATH,
    "current_paper_expanded_6632": EXPANDED_6632_PATH,
    "original_frenchyluba_6963": ORIGINAL_6963_PATH,
}
input_manifest = pd.DataFrame([
    {
        "role": role,
        "filename": path.name,
        "path": str(path),
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size,
    }
    for role, path in input_files.items()
])
input_manifest.to_csv(AUD / "input_file_manifest.csv", index=False)

observed_frozen_hash = sha256_file(FROZEN_PATH)
if observed_frozen_hash != EXPECTED_FROZEN_SHA256:
    raise ValueError(
        "The V7.4 frozen-resource hash does not match the accepted experiment. "
        f"Expected {EXPECTED_FROZEN_SHA256}; observed {observed_frozen_hash}."
    )

frozen = pd.read_csv(FROZEN_PATH, low_memory=False)
additions = pd.read_csv(ADDITIONS_PATH, low_memory=False)
contextual = pd.read_csv(CONTEXT_PATH, low_memory=False)
expanded_6632 = pd.read_excel(EXPANDED_6632_PATH)
original_6963 = pd.read_excel(ORIGINAL_6963_PATH)

if len(frozen) != EXPECTED_FROZEN_ROWS:
    raise ValueError(f"Frozen resource has {len(frozen)} rows; expected {EXPECTED_FROZEN_ROWS}.")
if len(expanded_6632) != 6632:
    raise ValueError(f"expanded_lexicon.xlsx has {len(expanded_6632)} rows; expected 6,632.")
if len(original_6963) != 6963:
    raise ValueError(f"lexicon_6000 words.xlsx has {len(original_6963)} rows; expected 6,963.")

required_frozen_columns = set(CORE_SCHEMA_COLUMNS) | {
    "v2_candidate_id", "v2_provenance", "v2_validation_status",
    "v2_sentiment_basis", "v2_researcher_meaning", "v2_researcher_word_type"
}
missing_frozen_columns = required_frozen_columns - set(frozen.columns)
if missing_frozen_columns:
    raise ValueError(f"Frozen resource is missing columns: {sorted(missing_frozen_columns)}")

addition_words = additions["Shona"].map(normalise_text).tolist()
if len(additions) != EXPECTED_ADDITION_COUNT:
    raise ValueError(f"Expected five accepted additions; found {len(additions)}.")
if sorted(addition_words) != sorted(EXPECTED_ADDITION_WORDS):
    raise ValueError(
        "The accepted V7.4 additions differ from the fixed set. "
        f"Observed: {sorted(addition_words)}"
    )

# Every addition must occur exactly once in the frozen resource.
frozen_norm_shona = frozen["Shona"].map(normalise_text)
addition_index = {}
for word in EXPECTED_ADDITION_WORDS:
    indices = frozen.index[frozen_norm_shona.eq(word)].tolist()
    if len(indices) != 1:
        raise ValueError(f"Expected exactly one frozen row for {word!r}; found {len(indices)}.")
    addition_index[word] = indices[0]

# Merge the researcher and contextual meanings without changing the frozen decisions.
contextual_key = contextual.rename(columns={"candidate_id": "v2_candidate_id", "word": "Shona"}).copy()
contextual_key["Shona"] = contextual_key["Shona"].map(normalise_text)
additions_meta = additions.copy()
additions_meta["Shona"] = additions_meta["Shona"].map(normalise_text)
additions_meta = additions_meta.merge(
    contextual_key[
        [
            "v2_candidate_id", "Shona", "contextual_english_meaning",
            "contextual_word_type", "contextual_lexical_sentiment",
            "contextual_corpus_association", "contextual_confidence"
        ]
    ],
    on=["v2_candidate_id", "Shona"],
    how="left",
    validate="one_to_one",
)

# Nature remains compatible with the existing FrenchyLuba inventory.
NATURE_MAP = {
    "NOUN": "Nom",
    "VERB": "Verbe",
    "ADJECTIVE": "Adjectif",
    "ADVERB": "Mot",
    "INTERJECTION": "Mot",
    "IDIOM": "Mot",
    "PHRASE": "Mot",
}
additions_meta["Nature_V2"] = additions_meta["v2_researcher_word_type"].astype(str).str.upper().map(NATURE_MAP)
if additions_meta["Nature_V2"].isna().any():
    unresolved = additions_meta.loc[additions_meta["Nature_V2"].isna(), ["Shona", "v2_researcher_word_type"]]
    raise ValueError("A Nature tag could not be mapped:\n" + unresolved.to_string(index=False))

additions_meta.to_csv(TAB / "five_additions_source_metadata.csv", index=False)

print("Frozen V7.4 rows:", len(frozen))
print("Frozen SHA-256 verified:", observed_frozen_hash)
print("Accepted additions:", ", ".join(EXPECTED_ADDITION_WORDS))
print("Current-paper resources: 6,632-row expanded lexicon and 6,963-row FrenchyLuba")

Frozen V7.4 rows: 6620
Frozen SHA-256 verified: 0cbfba6ea4c3e1bd84bc1fed31c364ab899e9c6a8e68314e10825bc070e74380
Accepted additions: vana, yekuroja, yekurara, kutengeswa, ndinoda
Current-paper resources: 6,632-row expanded lexicon and 6,963-row FrenchyLuba


In [4]:
# Fresh Google Translate generation: direct Shona route plus English-pivot cross-check


def load_cache(path, columns):
    if path.exists():
        cache = pd.read_csv(path, dtype=str, keep_default_na=False)
        for column in columns:
            if column not in cache.columns:
                cache[column] = ""
        return cache[columns]
    return pd.DataFrame(columns=columns)


def save_cache(cache, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    cache.to_csv(path, index=False)


cache_columns = [
    "source_text", "source_language", "target_language", "translated_text",
    "status", "error"
]
google_cache = load_cache(TRANSLATION_CACHE_PATH, cache_columns)
google_attempt_rows = []

# deep-translator exposes a runtime language dictionary; save it for the audit.
try:
    raw_supported = GoogleTranslator(source="auto", target="en").get_supported_languages(as_dict=True)
    google_supported = {str(k).casefold(): str(v) for k, v in raw_supported.items()}
except Exception as exc:
    google_supported = {}
    google_attempt_rows.append({
        "word": "", "field": "", "route": "runtime_language_list",
        "source_text": "", "source_language": "", "target_language": "",
        "translated_text": "", "status": "language_list_error",
        "error": f"{type(exc).__name__}: {exc}",
    })

pd.DataFrame(
    [{"language_name": name, "language_code": code} for name, code in sorted(google_supported.items())]
).to_csv(AUD / "google_runtime_supported_languages.csv", index=False)


def resolve_google_language(preferred_code, names):
    values = set(google_supported.values())
    if preferred_code in values:
        return preferred_code
    for name in names:
        if name.casefold() in google_supported:
            return google_supported[name.casefold()]
    # Attempt the preferred code even if the runtime list was unavailable.
    return preferred_code


SHONA_CODE = resolve_google_language(
    SHONA_LANGUAGE["preferred_code"], SHONA_LANGUAGE["names"]
)
RESOLVED_TARGET_CODES = {
    field: resolve_google_language(config["preferred_code"], config["names"])
    for field, config in GOOGLE_TARGETS.items()
}


def translation_usable(source_text, translated_text):
    if not nonblank(translated_text):
        return False
    # An unchanged result is retained in the audit but is not accepted as a translation.
    return normalise_compare(source_text) != normalise_compare(translated_text)


def google_translate_cached(source_text, source_language, target_language, word, field, route):
    global google_cache
    source_text = safe_string(source_text)
    if not source_text:
        result = {
            "word": word, "field": field, "route": route,
            "source_text": source_text, "source_language": source_language,
            "target_language": target_language, "translated_text": "",
            "status": "empty_source", "error": "",
        }
        google_attempt_rows.append(result)
        return result

    match = google_cache[
        google_cache["source_text"].eq(source_text)
        & google_cache["source_language"].eq(source_language)
        & google_cache["target_language"].eq(target_language)
    ]
    if len(match):
        cached = match.iloc[-1]
        result = {
            "word": word, "field": field, "route": route,
            "source_text": source_text, "source_language": source_language,
            "target_language": target_language,
            "translated_text": cached["translated_text"],
            "status": "cache_" + cached["status"], "error": cached["error"],
        }
        google_attempt_rows.append(result)
        return result

    last_error = ""
    translated = ""
    status = "failed"
    for attempt in range(1, GOOGLE_MAX_RETRIES + 1):
        try:
            translated = safe_string(
                GoogleTranslator(source=source_language, target=target_language).translate(source_text)
            )
            status = "success" if translation_usable(source_text, translated) else "unusable_unchanged_or_blank"
            if status == "success":
                break
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        if attempt < GOOGLE_MAX_RETRIES:
            time.sleep(GOOGLE_SLEEP_SECONDS * attempt)

    cache_row = pd.DataFrame([{
        "source_text": source_text,
        "source_language": source_language,
        "target_language": target_language,
        "translated_text": translated,
        "status": status,
        "error": last_error,
    }])
    google_cache = pd.concat([google_cache, cache_row], ignore_index=True)
    save_cache(google_cache, TRANSLATION_CACHE_PATH)

    result = {
        "word": word, "field": field, "route": route,
        "source_text": source_text, "source_language": source_language,
        "target_language": target_language, "translated_text": translated,
        "status": status, "error": last_error,
    }
    google_attempt_rows.append(result)
    return result


selection_rows = []
for _, row in additions_meta.sort_values("v2_candidate_id").iterrows():
    word = row["Shona"]
    researcher_meaning = safe_string(row.get("v2_researcher_meaning"))
    contextual_meaning = safe_string(row.get("contextual_english_meaning"))

    # Fresh Shona -> English is the pivot. Researcher/contextual meanings are comparisons/fallbacks.
    english_attempt = google_translate_cached(
        word, SHONA_CODE, RESOLVED_TARGET_CODES["English"], word, "English", "direct_shona"
    )
    google_english = english_attempt["translated_text"] if english_attempt["status"].endswith("success") else ""
    if nonblank(google_english):
        final_english = google_english
        english_source = "google_direct_shona"
    elif nonblank(researcher_meaning):
        final_english = researcher_meaning
        english_source = "researcher_meaning_fallback"
    elif nonblank(contextual_meaning):
        final_english = contextual_meaning
        english_source = "contextual_meaning_fallback"
    else:
        final_english = ""
        english_source = "unresolved"

    selection = {
        "candidate_id": row["v2_candidate_id"],
        "word": word,
        "researcher_meaning": researcher_meaning,
        "contextual_english_meaning": contextual_meaning,
        "English": final_english,
        "English_source": english_source,
        "Nature": row["Nature_V2"],
        "Nature_source": "v2_researcher_word_type_mapping",
    }

    for field in ["French", "Zulu", "Afrikaans", "Sepedi", "Xhosa"]:
        target_code = RESOLVED_TARGET_CODES[field]
        direct = google_translate_cached(
            word, SHONA_CODE, target_code, word, field, "direct_shona"
        )
        pivot = google_translate_cached(
            final_english, RESOLVED_TARGET_CODES["English"], target_code,
            word, field, "english_pivot"
        )
        direct_value = direct["translated_text"] if direct["status"].endswith("success") else ""
        pivot_value = pivot["translated_text"] if pivot["status"].endswith("success") else ""

        if nonblank(direct_value):
            final_value = direct_value
            selected_source = "google_direct_shona"
        elif nonblank(pivot_value):
            final_value = pivot_value
            selected_source = "google_english_pivot_fallback"
        else:
            final_value = ""
            selected_source = "unresolved"

        agreement = bool(
            nonblank(direct_value)
            and nonblank(pivot_value)
            and normalise_compare(direct_value) == normalise_compare(pivot_value)
        )
        selection.update({
            field: final_value,
            f"{field}_source": selected_source,
            f"{field}_direct_google": direct_value,
            f"{field}_pivot_google": pivot_value,
            f"{field}_direct_pivot_agree": agreement,
        })

    selection_rows.append(selection)

fresh_selections = pd.DataFrame(selection_rows)
google_attempts = pd.DataFrame(google_attempt_rows)
google_attempts.to_csv(TAB / "fresh_google_translation_attempts.csv", index=False)
fresh_selections.to_csv(TAB / "fresh_google_translation_summary.csv", index=False)

print("Fresh Google translation rows:", len(fresh_selections))
print("Resolved Google target codes:", RESOLVED_TARGET_CODES)

Fresh Google translation rows: 5
Resolved Google target codes: {'English': 'en', 'French': 'fr', 'Zulu': 'zu', 'Afrikaans': 'af', 'Sepedi': 'nso', 'Xhosa': 'xh'}


In [5]:
# Search the comparison lexicons for multilingual and Ciluba evidence


def normalised_column(df, candidates):
    lower_map = {str(column).casefold(): column for column in df.columns}
    for candidate in candidates:
        if candidate.casefold() in lower_map:
            return lower_map[candidate.casefold()]
    return None


# Build a comparison-only legacy table. These rows do not overwrite fresh Google values.
legacy_sources = []
for source_name, frame in [
    ("current_paper_expanded_6632", expanded_6632),
    ("original_frenchyluba_6963", original_6963),
]:
    english_col = normalised_column(frame, ["English"])
    french_col = normalised_column(frame, ["French", "FRANCAIS"])
    ciluba_col = normalised_column(frame, ["CILUBA"])
    nature_col = normalised_column(frame, ["Nature"])
    for row_index, source_row in frame.iterrows():
        legacy_sources.append({
            "source_resource": source_name,
            "source_row_1_based": int(row_index + 1),
            "CILUBA": safe_string(source_row.get(ciluba_col)) if ciluba_col else "",
            "French": safe_string(source_row.get(french_col)) if french_col else "",
            "English": safe_string(source_row.get(english_col)) if english_col else "",
            "Nature": safe_string(source_row.get(nature_col)) if nature_col else "",
        })
legacy_table = pd.DataFrame(legacy_sources)
legacy_table["_english_norm"] = legacy_table["English"].map(normalise_compare)
legacy_table["_french_norm"] = legacy_table["French"].map(normalise_compare)

match_rows = []
for _, candidate in fresh_selections.iterrows():
    english_query = normalise_compare(candidate["English"])
    french_query = normalise_compare(candidate["French"])
    scored_rows = []
    for _, legacy in legacy_table.iterrows():
        english_score = (
            float(fuzz.WRatio(english_query, legacy["_english_norm"]))
            if english_query and legacy["_english_norm"] else 0.0
        )
        french_score = (
            float(fuzz.WRatio(french_query, legacy["_french_norm"]))
            if french_query and legacy["_french_norm"] else 0.0
        )
        exact_english = bool(english_query and english_query == legacy["_english_norm"])
        exact_french = bool(french_query and french_query == legacy["_french_norm"])
        score = max(english_score, french_score)
        if score <= 0:
            continue
        scored_rows.append({
            "candidate_id": candidate["candidate_id"],
            "word": candidate["word"],
            "fresh_English": candidate["English"],
            "fresh_French": candidate["French"],
            "source_resource": legacy["source_resource"],
            "source_row_1_based": legacy["source_row_1_based"],
            "legacy_CILUBA": legacy["CILUBA"],
            "legacy_French": legacy["French"],
            "legacy_English": legacy["English"],
            "legacy_Nature": legacy["Nature"],
            "english_similarity": english_score,
            "french_similarity": french_score,
            "max_similarity": score,
            "exact_english_match": exact_english,
            "exact_french_match": exact_french,
        })
    scored_rows = sorted(
        scored_rows,
        key=lambda item: (
            item["exact_english_match"] or item["exact_french_match"],
            item["max_similarity"],
            nonblank(item["legacy_CILUBA"]),
        ),
        reverse=True,
    )[:LEGACY_MATCHES_PER_WORD]
    match_rows.extend(scored_rows)

concept_matches = pd.DataFrame(match_rows)
concept_matches.to_csv(TAB / "current_paper_lexicon_concept_matches.csv", index=False)

# A compact comparison table for the fresh supported-language outputs.
comparison_rows = []
for _, candidate in fresh_selections.iterrows():
    top = concept_matches[concept_matches["word"].eq(candidate["word"])].head(1)
    if len(top):
        legacy = top.iloc[0]
        comparison_rows.append({
            "candidate_id": candidate["candidate_id"],
            "word": candidate["word"],
            "fresh_English": candidate["English"],
            "fresh_French": candidate["French"],
            "top_legacy_resource": legacy["source_resource"],
            "top_legacy_row_1_based": legacy["source_row_1_based"],
            "top_legacy_English": legacy["legacy_English"],
            "top_legacy_French": legacy["legacy_French"],
            "top_legacy_CILUBA": legacy["legacy_CILUBA"],
            "top_similarity": legacy["max_similarity"],
            "exact_legacy_concept_match": bool(
                legacy["exact_english_match"] or legacy["exact_french_match"]
            ),
        })
    else:
        comparison_rows.append({
            "candidate_id": candidate["candidate_id"], "word": candidate["word"],
            "fresh_English": candidate["English"], "fresh_French": candidate["French"],
            "top_legacy_resource": "", "top_legacy_row_1_based": "",
            "top_legacy_English": "", "top_legacy_French": "",
            "top_legacy_CILUBA": "", "top_similarity": 0.0,
            "exact_legacy_concept_match": False,
        })
legacy_comparison = pd.DataFrame(comparison_rows)
legacy_comparison.to_csv(TAB / "fresh_translation_vs_current_paper_comparison.csv", index=False)

print("Legacy concept-match rows saved:", len(concept_matches))

Legacy concept-match rows saved: 40


In [6]:
# Ciluba hierarchy: Google first, then constrained FrenchyLuba-assisted GPT proposal


def get_openai_client():
    api_key = ""
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = safe_string(UserSecretsClient().get_secret(OPENAI_SECRET_NAME))
    except Exception:
        api_key = safe_string(os.environ.get(OPENAI_SECRET_NAME, ""))
    if not api_key:
        return None
    return OpenAI(api_key=api_key)


# Resolve any Ciluba/Tshiluba target explicitly exposed by the current Google runtime.
ciluba_targets = []
for language_name, language_code in google_supported.items():
    if any(candidate in language_name for candidate in CILUBA_RUNTIME_NAMES):
        ciluba_targets.append((language_name, language_code))
for code_to_try in CILUBA_RUNTIME_CODES_TO_TRY:
    if code_to_try not in [code for _, code in ciluba_targets]:
        ciluba_targets.append(("code_attempt", code_to_try))

ciluba_cache_columns = [
    "candidate_id", "word", "ciluba_proposal", "confidence", "evidence_basis",
    "supporting_rows", "reason", "model", "status"
]
ciluba_gpt_cache = load_cache(CILUBA_CACHE_PATH, ciluba_cache_columns)
client = get_openai_client()
ciluba_rows = []


def call_ciluba_gpt(candidate, evidence_rows):
    global ciluba_gpt_cache
    candidate_id = candidate["candidate_id"]
    word = candidate["word"]
    existing = ciluba_gpt_cache[
        ciluba_gpt_cache["candidate_id"].eq(candidate_id)
        & ciluba_gpt_cache["word"].eq(word)
    ]
    if len(existing):
        return existing.iloc[-1].to_dict()

    if client is None:
        return {
            "candidate_id": candidate_id, "word": word, "ciluba_proposal": "",
            "confidence": "", "evidence_basis": "unresolved_no_api_key",
            "supporting_rows": "", "reason": "OPENAI_API_KEY was unavailable.",
            "model": OPENAI_MODEL, "status": "unresolved",
        }

    evidence_payload = []
    for _, evidence in evidence_rows.iterrows():
        if not nonblank(evidence.get("legacy_CILUBA")):
            continue
        evidence_payload.append({
            "source_resource": evidence["source_resource"],
            "source_row_1_based": int(evidence["source_row_1_based"]),
            "ciluba": safe_string(evidence["legacy_CILUBA"]),
            "french": safe_string(evidence["legacy_French"]),
            "english": safe_string(evidence["legacy_English"]),
            "similarity": float(evidence["max_similarity"]),
        })

    request_payload = {
        "shona": word,
        "fresh_english": candidate["English"],
        "fresh_french": candidate["French"],
        "researcher_meaning": candidate["researcher_meaning"],
        "current_paper_lexicon_evidence": evidence_payload,
    }
    system_text = (
        "You are proposing one possible Ciluba/Tshiluba lexical equivalent for a multilingual "
        "sentiment-lexicon audit. Use only the supplied Shona/English/French meanings and the "
        "FrenchyLuba evidence rows. Do not invent supporting citations. If the evidence is "
        "insufficient, return an empty proposal and evidence_basis='unresolved'. Return JSON only."
    )
    user_text = (
        "Return exactly these fields: ciluba_proposal (string), confidence (number 0 to 1), "
        "evidence_basis (legacy_supported, lexicon_assisted_proposal, or unresolved), "
        "supporting_rows (array of source_resource:row identifiers), and reason (short string).\n\n"
        + json.dumps(request_payload, ensure_ascii=False)
    )

    last_error = ""
    for attempt in range(1, OPENAI_MAX_RETRIES + 1):
        try:
            response = client.responses.create(
                model=OPENAI_MODEL,
                input=[
                    {"role": "system", "content": system_text},
                    {"role": "user", "content": user_text},
                ],
            )
            parsed = parse_json_object(response.output_text)
            proposal = safe_string(parsed.get("ciluba_proposal"))
            try:
                confidence = float(parsed.get("confidence", 0.0))
            except Exception:
                confidence = 0.0
            confidence = float(np.clip(confidence, 0.0, 1.0))
            evidence_basis = safe_string(parsed.get("evidence_basis")) or "unresolved"
            supporting = parsed.get("supporting_rows", [])
            if isinstance(supporting, list):
                supporting = ";".join(safe_string(value) for value in supporting)
            else:
                supporting = safe_string(supporting)
            status = "completed" if proposal else "unresolved"
            record = {
                "candidate_id": candidate_id, "word": word,
                "ciluba_proposal": proposal, "confidence": confidence,
                "evidence_basis": evidence_basis, "supporting_rows": supporting,
                "reason": safe_string(parsed.get("reason")),
                "model": OPENAI_MODEL, "status": status,
            }
            ciluba_gpt_cache = pd.concat(
                [ciluba_gpt_cache, pd.DataFrame([record])], ignore_index=True
            )
            save_cache(ciluba_gpt_cache, CILUBA_CACHE_PATH)
            return record
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            if attempt < OPENAI_MAX_RETRIES:
                time.sleep(OPENAI_SLEEP_SECONDS * attempt)

    return {
        "candidate_id": candidate_id, "word": word, "ciluba_proposal": "",
        "confidence": "", "evidence_basis": "unresolved_api_error",
        "supporting_rows": "", "reason": last_error,
        "model": OPENAI_MODEL, "status": "unresolved",
    }


for _, candidate in fresh_selections.iterrows():
    word = candidate["word"]
    google_ciluba_value = ""
    google_ciluba_route = ""
    google_ciluba_status = "unsupported_or_failed"
    google_ciluba_errors = []

    # Try every Ciluba/Tshiluba target exposed or explicitly attempted at runtime.
    for target_name, target_code in ciluba_targets:
        direct = google_translate_cached(
            word, SHONA_CODE, target_code, word, "CILUBA", f"direct_shona_{target_name}"
        )
        if direct["status"].endswith("success"):
            google_ciluba_value = direct["translated_text"]
            google_ciluba_route = f"google_direct_shona:{target_code}"
            google_ciluba_status = "google_machine_translation_unverified"
            break
        google_ciluba_errors.append(direct["error"] or direct["status"])

        pivot = google_translate_cached(
            candidate["English"], RESOLVED_TARGET_CODES["English"], target_code,
            word, "CILUBA", f"english_pivot_{target_name}"
        )
        if pivot["status"].endswith("success"):
            google_ciluba_value = pivot["translated_text"]
            google_ciluba_route = f"google_english_pivot:{target_code}"
            google_ciluba_status = "google_machine_translation_unverified"
            break
        google_ciluba_errors.append(pivot["error"] or pivot["status"])

    candidate_evidence = concept_matches[
        concept_matches["word"].eq(word)
        & concept_matches["legacy_CILUBA"].map(nonblank)
    ].copy()
    candidate_evidence = candidate_evidence.sort_values(
        ["exact_english_match", "exact_french_match", "max_similarity"],
        ascending=[False, False, False],
    ).head(LEGACY_MATCHES_PER_WORD)

    if nonblank(google_ciluba_value):
        final_ciluba = google_ciluba_value
        final_method = google_ciluba_route
        final_status = google_ciluba_status
        gpt_record = {}
    else:
        gpt_record = call_ciluba_gpt(candidate, candidate_evidence)
        final_ciluba = safe_string(gpt_record.get("ciluba_proposal"))
        if nonblank(final_ciluba):
            final_method = "gpt_frenchyluba_assisted"
            final_status = "gpt_lexicon_assisted_provisional"
        else:
            final_method = "unresolved"
            final_status = "unresolved"

    ciluba_rows.append({
        "candidate_id": candidate["candidate_id"],
        "word": word,
        "google_runtime_target_attempts": ";".join(
            f"{name}:{code}" for name, code in ciluba_targets
        ),
        "google_ciluba_value": google_ciluba_value,
        "google_ciluba_route": google_ciluba_route,
        "google_ciluba_status": google_ciluba_status,
        "google_errors": " | ".join(error for error in google_ciluba_errors if error),
        "legacy_evidence_rows": int(len(candidate_evidence)),
        "gpt_ciluba_proposal": safe_string(gpt_record.get("ciluba_proposal")),
        "gpt_confidence": gpt_record.get("confidence", ""),
        "gpt_evidence_basis": safe_string(gpt_record.get("evidence_basis")),
        "gpt_supporting_rows": safe_string(gpt_record.get("supporting_rows")),
        "gpt_reason": safe_string(gpt_record.get("reason")),
        "final_CILUBA": final_ciluba,
        "final_CILUBA_method": final_method,
        "final_CILUBA_status": final_status,
    })

ciluba_audit = pd.DataFrame(ciluba_rows)
ciluba_audit.to_csv(TAB / "ciluba_translation_audit.csv", index=False)

print("Ciluba rows completed:", int(ciluba_audit["final_CILUBA"].map(nonblank).sum()), "/", EXPECTED_ADDITION_COUNT)
print("Ciluba method counts:\n", ciluba_audit["final_CILUBA_method"].value_counts(dropna=False))

/tmp/ipykernel_58/411166655.py:117: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ciluba_gpt_cache = pd.concat(


Ciluba rows completed: 5 / 5
Ciluba method counts:
 final_CILUBA_method
gpt_frenchyluba_assisted    5
Name: count, dtype: int64


In [7]:
# Assemble the five multilingual rows and prove frozen-field invariance

selections = fresh_selections.merge(
    ciluba_audit[
        [
            "candidate_id", "word", "final_CILUBA", "final_CILUBA_method",
            "final_CILUBA_status", "gpt_confidence", "gpt_reason"
        ]
    ],
    on=["candidate_id", "word"],
    how="left",
    validate="one_to_one",
)
selections = selections.rename(columns={"final_CILUBA": "CILUBA"})

# Build one provenance row for every language field.
provenance_rows = []
for _, row in selections.iterrows():
    provenance_rows.append({
        "candidate_id": row["candidate_id"], "word": row["word"],
        "field": "English", "final_value": row["English"],
        "primary_method": row["English_source"],
        "comparison_method": "researcher_and_contextual_meanings",
        "validation_status": "fresh_v2_machine_translation_or_documented_fallback",
    })
    for field in ["French", "Zulu", "Afrikaans", "Sepedi", "Xhosa"]:
        provenance_rows.append({
            "candidate_id": row["candidate_id"], "word": row["word"],
            "field": field, "final_value": row[field],
            "primary_method": row[f"{field}_source"],
            "comparison_method": "fresh_google_direct_vs_english_pivot",
            "validation_status": (
                "direct_and_pivot_agree" if bool(row[f"{field}_direct_pivot_agree"])
                else "direct_primary_pivot_disagreement_or_missing"
            ),
        })
    provenance_rows.append({
        "candidate_id": row["candidate_id"], "word": row["word"],
        "field": "CILUBA", "final_value": row["CILUBA"],
        "primary_method": row["final_CILUBA_method"],
        "comparison_method": "current_paper_frenchyluba_concept_evidence",
        "validation_status": row["final_CILUBA_status"],
    })
    provenance_rows.append({
        "candidate_id": row["candidate_id"], "word": row["word"],
        "field": "Nature", "final_value": row["Nature"],
        "primary_method": row["Nature_source"],
        "comparison_method": "existing_frenchyluba_nature_inventory",
        "validation_status": "schema_compatible_researcher_word_type_mapping",
    })
translation_provenance = pd.DataFrame(provenance_rows)
translation_provenance.to_csv(TAB / "translation_provenance_long.csv", index=False)

release_schema = frozen.copy()
for _, selected in selections.iterrows():
    row_index = addition_index[selected["word"]]
    for field in ENRICHMENT_FIELDS:
        release_schema.at[row_index, field] = selected[field]

# The experimental identity is every original column except the eight enrichment fields.
protected_columns = [column for column in frozen.columns if column not in ENRICHMENT_FIELDS]
invariance_rows = []
for column in frozen.columns:
    identical = frozen[column].astype("string").fillna("").equals(
        release_schema[column].astype("string").fillna("")
    )
    invariance_rows.append({
        "column": column,
        "allowed_to_change": column in ENRICHMENT_FIELDS,
        "identical_to_frozen": identical,
    })
invariance = pd.DataFrame(invariance_rows)
invariance.to_csv(AUD / "release_vs_frozen_invariance.csv", index=False)

protected_changed = invariance[
    (~invariance["allowed_to_change"]) & (~invariance["identical_to_frozen"])
]
if len(protected_changed):
    raise AssertionError(
        "Protected frozen columns changed: " + ", ".join(protected_changed["column"])
    )

# Only the five accepted additions may change in allowed columns.
changed_rows = []
for index in frozen.index:
    fields = [
        field for field in ENRICHMENT_FIELDS
        if normalise_text(frozen.at[index, field]) != normalise_text(release_schema.at[index, field])
    ]
    if fields:
        changed_rows.append({
            "row_index_0_based": int(index),
            "Shona": release_schema.at[index, "Shona"],
            "changed_fields": ";".join(fields),
        })
changed_rows_df = pd.DataFrame(changed_rows)
changed_rows_df.to_csv(AUD / "release_changed_rows_and_fields.csv", index=False)
if sorted(changed_rows_df["Shona"].map(normalise_text).tolist()) != sorted(EXPECTED_ADDITION_WORDS):
    raise AssertionError("Rows outside the five accepted V7.4 additions were modified.")

# Save a schema-compatible copy and an audited copy with translation metadata.
schema_release_path = ROOT_OUT / "FrenchyShona_V2_v7_4_multilingual_release_core13.csv"
release_schema[CORE_SCHEMA_COLUMNS].to_csv(schema_release_path, index=False)

full_release_path = ROOT_OUT / "FrenchyShona_V2_v7_4_multilingual_release.csv"
audited_release = release_schema.copy()
audited_release["v2_multilingual_enrichment_version"] = pd.NA
audited_release["v2_ciluba_translation_status"] = pd.NA
audited_release["v2_ciluba_translation_method"] = pd.NA
for _, selected in selections.iterrows():
    row_index = addition_index[selected["word"]]
    audited_release.at[row_index, "v2_multilingual_enrichment_version"] = NOTEBOOK_VERSION
    audited_release.at[row_index, "v2_ciluba_translation_status"] = selected["final_CILUBA_status"]
    audited_release.at[row_index, "v2_ciluba_translation_method"] = selected["final_CILUBA_method"]
audited_release.to_csv(full_release_path, index=False)

five_release = release_schema.loc[
    release_schema["Shona"].map(normalise_text).isin(EXPECTED_ADDITION_WORDS),
    CORE_SCHEMA_COLUMNS,
].copy()
five_release.to_csv(TAB / "five_additions_multilingual_release.csv", index=False)
selections.to_csv(TAB / "five_additions_translation_selection_audit.csv", index=False)

print("Schema-compatible multilingual release:", schema_release_path)
print("Audited multilingual release:", full_release_path)
print("Protected frozen fields unchanged: PASS")

Schema-compatible multilingual release: /kaggle/working/FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment/FrenchyShona_V2_v7_4_multilingual_release_core13.csv
Audited multilingual release: /kaggle/working/FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment/FrenchyShona_V2_v7_4_multilingual_release.csv
Protected frozen fields unchanged: PASS


In [8]:
# Completion tables, gates, workbook and research figures

completion_rows = []
for field in ["English", "French", "Zulu", "Afrikaans", "Sepedi", "Xhosa", "CILUBA", "Nature"]:
    completion_rows.append({
        "field": field,
        "filled": int(selections[field].map(nonblank).sum()),
        "total": EXPECTED_ADDITION_COUNT,
        "completion_rate": float(selections[field].map(nonblank).mean()),
    })
completion = pd.DataFrame(completion_rows)
completion.to_csv(TAB / "translation_completion_summary.csv", index=False)

agreement_rows = []
for field in ["French", "Zulu", "Afrikaans", "Sepedi", "Xhosa"]:
    both = selections[
        selections[f"{field}_direct_google"].map(nonblank)
        & selections[f"{field}_pivot_google"].map(nonblank)
    ]
    agreement_rows.append({
        "field": field,
        "both_routes_available": int(len(both)),
        "direct_pivot_agreements": int(both[f"{field}_direct_pivot_agree"].sum()),
        "agreement_rate_when_both_available": (
            float(both[f"{field}_direct_pivot_agree"].mean()) if len(both) else np.nan
        ),
    })
google_agreement = pd.DataFrame(agreement_rows)
google_agreement.to_csv(TAB / "fresh_google_direct_pivot_agreement.csv", index=False)

supported_complete = bool(
    selections[["English", "French", "Zulu", "Afrikaans", "Sepedi", "Xhosa"]]
    .apply(lambda column: column.map(nonblank)).all(axis=None)
)
nature_complete = bool(selections["Nature"].map(nonblank).all())
ciluba_attempt_documented = bool(len(ciluba_audit) == EXPECTED_ADDITION_COUNT)
ciluba_values_present = bool(selections["CILUBA"].map(nonblank).all())
protected_invariance = bool(len(protected_changed) == 0)
exact_five_changed = bool(
    sorted(changed_rows_df["Shona"].map(normalise_text).tolist())
    == sorted(EXPECTED_ADDITION_WORDS)
)

# Part 2 uses the frozen Shona resource; provisional Ciluba does not alter the model evidence.
part2_start_allowed = bool(
    observed_frozen_hash == EXPECTED_FROZEN_SHA256
    and len(frozen) == EXPECTED_FROZEN_ROWS
    and supported_complete
    and nature_complete
    and ciluba_attempt_documented
    and protected_invariance
    and exact_five_changed
)
part2_gate = pd.DataFrame([{
    "frozen_v7_4_hash_verified": observed_frozen_hash == EXPECTED_FROZEN_SHA256,
    "frozen_rows_verified": len(frozen) == EXPECTED_FROZEN_ROWS,
    "five_additions_verified": sorted(addition_words) == sorted(EXPECTED_ADDITION_WORDS),
    "fresh_supported_language_fields_complete": supported_complete,
    "nature_fields_complete": nature_complete,
    "ciluba_attempt_documented_for_all_five": ciluba_attempt_documented,
    "ciluba_value_present_for_all_five": ciluba_values_present,
    "protected_frozen_fields_unchanged": protected_invariance,
    "changed_rows_are_exactly_the_five_additions": exact_five_changed,
    "part2_start_allowed_after_result_review": part2_start_allowed,
}])
part2_gate.to_csv(ROOT_OUT / "PART2_START_GATE.csv", index=False)

ciluba_requires_review = bool(
    selections["final_CILUBA_status"].astype(str).str.contains(
        "provisional|unverified|unresolved", case=False, regex=True
    ).any()
)
public_release_ready = bool(
    supported_complete
    and nature_complete
    and ciluba_values_present
    and protected_invariance
    and not ciluba_requires_review
)
public_gate = pd.DataFrame([{
    "supported_language_fields_complete": supported_complete,
    "ciluba_values_present_for_all_five": ciluba_values_present,
    "ciluba_contains_provisional_unverified_or_unresolved_values": ciluba_requires_review,
    "protected_frozen_fields_unchanged": protected_invariance,
    "multilingual_public_release_ready_without_additional_language_review": public_release_ready,
    "interpretation": (
        "Ready for public release" if public_release_ready
        else "Experimental release is complete; Ciluba/native-language review remains required"
    ),
}])
public_gate.to_csv(ROOT_OUT / "MULTILINGUAL_PUBLIC_RELEASE_GATE.csv", index=False)

unresolved_or_provisional = translation_provenance[
    translation_provenance["validation_status"].astype(str).str.contains(
        "provisional|unverified|unresolved", case=False, regex=True
    )
].copy()
unresolved_or_provisional.to_csv(
    TAB / "provisional_or_unresolved_translation_fields.csv", index=False
)

# One audit workbook for the five additions.
workbook_path = ROOT_OUT / "FrenchyShona_V2_five_additions_multilingual_audit.xlsx"
with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
    five_release.to_excel(writer, sheet_name="five_additions_release", index=False)
    selections.to_excel(writer, sheet_name="translation_selection", index=False)
    google_attempts.to_excel(writer, sheet_name="google_attempts", index=False)
    google_agreement.to_excel(writer, sheet_name="google_route_agreement", index=False)
    concept_matches.to_excel(writer, sheet_name="legacy_concept_matches", index=False)
    ciluba_audit.to_excel(writer, sheet_name="ciluba_audit", index=False)
    translation_provenance.to_excel(writer, sheet_name="field_provenance", index=False)
    completion.to_excel(writer, sheet_name="completion", index=False)
    invariance.to_excel(writer, sheet_name="frozen_invariance", index=False)
    part2_gate.to_excel(writer, sheet_name="part2_start_gate", index=False)
    public_gate.to_excel(writer, sheet_name="public_release_gate", index=False)

# Figure 1: completion
plt.figure(figsize=(8.5, 4.8))
plt.bar(completion["field"], completion["filled"])
plt.ylim(0, EXPECTED_ADDITION_COUNT + 0.7)
plt.ylabel("Filled additions")
plt.title("Multilingual completion for the five V7.4 additions")
plt.xticks(rotation=30, ha="right")
for index, value in enumerate(completion["filled"]):
    plt.text(index, value + 0.08, f"{value}/{EXPECTED_ADDITION_COUNT}", ha="center")
plt.tight_layout()
plt.savefig(FIG / "fig_five_additions_translation_completion.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 2: Google direct/pivot agreement
plt.figure(figsize=(8, 4.8))
plt.bar(google_agreement["field"], google_agreement["direct_pivot_agreements"])
plt.ylim(0, EXPECTED_ADDITION_COUNT + 0.7)
plt.ylabel("Agreements")
plt.title("Fresh Google direct Shona vs English-pivot agreement")
plt.xticks(rotation=25, ha="right")
for index, value in enumerate(google_agreement["direct_pivot_agreements"]):
    total = google_agreement.iloc[index]["both_routes_available"]
    plt.text(index, value + 0.08, f"{int(value)}/{int(total)}", ha="center")
plt.tight_layout()
plt.savefig(FIG / "fig_google_direct_pivot_agreement.png", dpi=300, bbox_inches="tight")
plt.close()

# Figure 3: Ciluba method status
ciluba_method_counts = ciluba_audit["final_CILUBA_method"].value_counts(dropna=False)
plt.figure(figsize=(8.5, 4.8))
plt.bar(ciluba_method_counts.index.astype(str), ciluba_method_counts.values)
plt.ylabel("Additions")
plt.title("Ciluba translation method for the five V7.4 additions")
plt.xticks(rotation=30, ha="right")
for index, value in enumerate(ciluba_method_counts.values):
    plt.text(index, value + 0.05, str(int(value)), ha="center")
plt.tight_layout()
plt.savefig(FIG / "fig_ciluba_translation_methods.png", dpi=300, bbox_inches="tight")
plt.close()

print(part2_gate.to_string(index=False))
print(public_gate.to_string(index=False))

 frozen_v7_4_hash_verified  frozen_rows_verified  five_additions_verified  fresh_supported_language_fields_complete  nature_fields_complete  ciluba_attempt_documented_for_all_five  ciluba_value_present_for_all_five  protected_frozen_fields_unchanged  changed_rows_are_exactly_the_five_additions  part2_start_allowed_after_result_review
                      True                  True                     True                                      True                    True                                    True                               True                               True                                         True                                     True
 supported_language_fields_complete  ciluba_values_present_for_all_five  ciluba_contains_provisional_unverified_or_unresolved_values  protected_frozen_fields_unchanged  multilingual_public_release_ready_without_additional_language_review                                                                   interpretation
         

In [9]:
# Research-output inventory and ZIP archive

allowed_suffixes = {".csv", ".xlsx", ".png"}
output_records = []
for path in sorted(ROOT_OUT.rglob("*")):
    if path.is_file() and path.suffix.casefold() in allowed_suffixes:
        output_records.append({
            "relative_path": str(path.relative_to(ROOT_OUT)),
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        })
output_inventory = pd.DataFrame(output_records)
output_inventory.to_csv(ROOT_OUT / "output_inventory.csv", index=False)

# Recompute after adding the inventory itself.
output_records = []
for path in sorted(ROOT_OUT.rglob("*")):
    if path.is_file() and path.suffix.casefold() in allowed_suffixes:
        output_records.append({
            "relative_path": str(path.relative_to(ROOT_OUT)),
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        })
pd.DataFrame(output_records).to_csv(ROOT_OUT / "output_checksums.csv", index=False)

zip_name = "FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment_Results.zip"
zip_path = ROOT_OUT.parent / zip_name
zip_path.parent.mkdir(parents=True, exist_ok=True)
temporary_zip = zip_path.with_suffix(".tmp.zip")
if temporary_zip.exists():
    temporary_zip.unlink()

with zipfile.ZipFile(temporary_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ROOT_OUT.rglob("*")):
        if path.is_file() and path.suffix.casefold() in allowed_suffixes:
            archive.write(
                path,
                arcname=str(
                    Path("FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment")
                    / path.relative_to(ROOT_OUT)
                ),
            )

with zipfile.ZipFile(temporary_zip, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Results ZIP failed integrity check at {bad_member}")
    forbidden = [
        name for name in archive.namelist()
        if Path(name).suffix.casefold() not in allowed_suffixes
    ]
    if forbidden:
        raise RuntimeError(f"Unexpected non-research files in results ZIP: {forbidden[:10]}")

if zip_path.exists():
    zip_path.unlink()
temporary_zip.replace(zip_path)

print("\nRUN COMPLETE")
print("Results ZIP:", zip_path)
print("Part 2 start gate:", bool(part2_gate.iloc[0]["part2_start_allowed_after_result_review"]))
print("Public release gate:", bool(public_gate.iloc[0]["multilingual_public_release_ready_without_additional_language_review"]))


RUN COMPLETE
Results ZIP: /kaggle/working/FrenchyShona_V2_Part1B_Fresh_Multilingual_Enrichment_Results.zip
Part 2 start gate: True
Public release gate: False
Upload the results ZIP for assessment before beginning Part 2.
